<a href="https://colab.research.google.com/github/krishnasaicareer/codegen26/blob/main/notebooks/finetune_qwen.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Colab setup (run first)

**Code comes from GitHub**, not Drive:

`git clone https://github.com/krishnasaicareer/codegen26.git` → `/content/codegenQwen/codegen26`

Run in order: **Clone → PROJECT_ROOT → Inventory → Download → Install**


In [ ]:
# CELL A — Clone GitHub repo (ACTIVE — not commented)
import os
import shutil
import subprocess
from pathlib import Path

REPO_URL = "https://github.com/krishnasaicareer/codegen26.git"
BRANCH = "main"
TARGET = Path("/content/codegenQwen/codegen26")
# True wipes /content copy first. Prefer False if you already have a good clone.
FORCE_RECLONE = True
# Repo is public — leave False unless you need a private fork. A bad Colab
# GITHUB_TOKEN secret commonly causes git exit 128.
USE_GITHUB_TOKEN = False


def is_project(path: Path) -> bool:
    return (path / "requirements.txt").is_file() and (
        path / "data" / "scripts" / "preprocess_multitask.py"
    ).is_file()


def git_clone(url: str, dest: Path) -> None:
    cmd = ["git", "clone", "--depth", "1", "--branch", BRANCH, url, str(dest)]
    print("Running:", " ".join(cmd[:6]), REPO_URL, "->", dest)
    proc = subprocess.run(cmd, capture_output=True, text=True)
    if proc.returncode != 0:
        err = (proc.stderr or proc.stdout or "").strip()
        raise RuntimeError(
            f"git clone failed (exit {proc.returncode}).\n{err}\n"
            "If you see Authentication failed: clear/fix Colab Secret GITHUB_TOKEN, "
            "or set USE_GITHUB_TOKEN=False (public repo)."
        )


print("Cloning from GitHub into", TARGET)
TARGET.parent.mkdir(parents=True, exist_ok=True)

# If the kernel cwd is inside TARGET, deleting it makes git fail with:
#   fatal: Unable to read current working directory: No such file or directory
safe_cwd = Path("/content")
if not safe_cwd.exists():
    safe_cwd = Path.home()
os.chdir(safe_cwd)
print("cwd set to", Path.cwd())

if FORCE_RECLONE and TARGET.exists():
    print("Removing old copy...")
    shutil.rmtree(TARGET)

if not is_project(TARGET):
    clone_url = REPO_URL
    if USE_GITHUB_TOKEN:
        try:
            from google.colab import userdata  # type: ignore
            token = userdata.get("GITHUB_TOKEN")
            if token:
                clone_url = "https://{}@github.com/krishnasaicareer/codegen26.git".format(token)
                print("Using GITHUB_TOKEN")
        except Exception as e:
            print("GITHUB_TOKEN unused:", e)
    try:
        git_clone(clone_url, TARGET)
    except RuntimeError as e:
        # Fall back to anonymous public clone if a token URL failed.
        if clone_url != REPO_URL:
            print("Token clone failed; retrying public URL...\n", e)
            if TARGET.exists():
                shutil.rmtree(TARGET)
            git_clone(REPO_URL, TARGET)
        else:
            raise
else:
    print("Already present:", TARGET)

if not is_project(TARGET):
    raise RuntimeError("Clone failed: project markers missing in " + str(TARGET))

os.chdir(str(TARGET))
os.environ["PROJECT_ROOT"] = str(TARGET.resolve())
sha = subprocess.check_output(["git", "rev-parse", "--short", "HEAD"], cwd=str(TARGET), text=True).strip()
print("SUCCESS")
print("PROJECT_ROOT =", os.environ["PROJECT_ROOT"])
print("git HEAD =", sha)


In [2]:
# CELL B — Optional Drive data zip only (code is from GitHub above)
RUN_DATA_UNZIP = False

from pathlib import Path
import os

print("RUN_DATA_UNZIP =", RUN_DATA_UNZIP)
print("Skip unless you need to restore data/raw from Drive.")
if RUN_DATA_UNZIP:
    from google.colab import drive  # type: ignore
    import zipfile
    drive.mount("/content/drive", force_remount=False)
    root = Path(os.environ.get("PROJECT_ROOT", "/content/codegenQwen/codegen26"))
    zips = [
        "/content/drive/MyDrive/codegen26_data_raw.zip",
        "/content/drive/MyDrive/codegenQwen_data_raw.zip",
    ]
    zip_path = next((Path(p) for p in zips if Path(p).is_file()), None)
    if zip_path is None:
        raise FileNotFoundError(str(zips))
    dest = root / "data" / "raw"
    dest.mkdir(parents=True, exist_ok=True)
    with zipfile.ZipFile(zip_path) as zf:
        zf.extractall(dest)
    print("Extracted to", dest)


RUN_DATA_UNZIP = False
Skip unless you need to restore data/raw from Drive.


In [3]:
# CELL C — Set PROJECT_ROOT (run AFTER clone cell)
import os
import sys
from pathlib import Path

try:
    import google.colab  # type: ignore  # noqa: F401
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

DEFAULT = Path("/content/codegenQwen/codegen26")


def ok(root: Path) -> bool:
    return (root / "requirements.txt").is_file() and (
        root / "data" / "scripts" / "preprocess_multitask.py"
    ).is_file()


candidates = []
if os.environ.get("PROJECT_ROOT"):
    candidates.append(Path(os.environ["PROJECT_ROOT"]))
candidates.append(DEFAULT)
candidates.append(Path.cwd())
if Path.cwd().name == "notebooks":
    candidates.append(Path.cwd().parent)

print("Resolving PROJECT_ROOT (NO Drive mount required)...")
PROJECT_ROOT = None
for c in candidates:
    try:
        p = c.resolve()
    except OSError:
        continue
    print(" ", "OK" if ok(p) else "no", p)
    if ok(p):
        PROJECT_ROOT = p
        break

if PROJECT_ROOT is None:
    raise AssertionError(
        "PROJECT_ROOT missing. Run CELL A (GitHub clone) first. "
        "Expected /content/codegenQwen/codegen26"
    )

os.chdir(PROJECT_ROOT)
os.environ["PROJECT_ROOT"] = str(PROJECT_ROOT)
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

print("IN_COLAB:", IN_COLAB)
print("PROJECT_ROOT:", PROJECT_ROOT)


Resolving PROJECT_ROOT (NO Drive mount required)...
  OK /content/codegenQwen/codegen26
IN_COLAB: True
PROJECT_ROOT: /content/codegenQwen/codegen26


In [4]:
# CELL D — Raw data inventory
from pathlib import Path

RAW = PROJECT_ROOT / "data" / "raw"
REQUIRED = {
    "avatar_tc": RAW / "avatar_tc",
    "classeval_t": RAW / "classeval_t",
    "design_patterns_solid": RAW / "design_patterns_solid",
}
OPTIONAL = {"codocbench": RAW / "codocbench"}

print("PROJECT_ROOT:", PROJECT_ROOT)
for name, path in {**REQUIRED, **OPTIONAL}.items():
    status = "OK" if path.is_dir() else "MISSING"
    print(f"  {status:8} {name:24} {path}")
print("HF: MBPP (NL2Py), DocuMint (Code2Doc)")


PROJECT_ROOT: /content/codegenQwen/codegen26
  OK       avatar_tc                /content/codegenQwen/codegen26/data/raw/avatar_tc
  OK       classeval_t              /content/codegenQwen/codegen26/data/raw/classeval_t
  OK       design_patterns_solid    /content/codegenQwen/codegen26/data/raw/design_patterns_solid
  MISSING  codocbench               /content/codegenQwen/codegen26/data/raw/codocbench
HF: MBPP (NL2Py), DocuMint (Code2Doc)


In [5]:
# CELL E — Optional codocbench download
RUN_DOWNLOAD = True
import os
import subprocess

raw = PROJECT_ROOT / "data" / "raw"
if RUN_DOWNLOAD and not (raw / "codocbench").exists():
    env = os.environ.copy()
    env["DOWNLOAD_SQL"] = "0"
    print("Running data/scripts/download.sh for codocbench...")
    subprocess.check_call(["bash", "data/scripts/download.sh"], cwd=str(PROJECT_ROOT), env=env)
elif RUN_DOWNLOAD:
    print("codocbench present — skip")
else:
    print("skipped")


Running data/scripts/download.sh for codocbench...


In [6]:
# Install dependencies (Colab only). Colab already ships a CUDA-enabled
# PyTorch, so we install the rest of the stack here. trl/transformers/peft are
# pinned to versions compatible with this project's training API
# (SFTConfig(max_length=, dataset_text_field=) + SFTTrainer(processing_class=)).
if IN_COLAB:
    %pip install -q \
        "transformers>=4.46.0" \
        "datasets>=2.18.0" \
        "peft>=0.13.0" \
        "accelerate>=0.34.0" \
        "trl>=0.12.0" \
        sentencepiece \
        "evaluate>=0.4.0" \
        "sacrebleu>=2.4.0" \
        "bert-score>=0.3.13" \
        "codebleu[all]" \
        "nltk>=3.8.0" \
        pyyaml
    # If CodeBLEU returns "an integer is required", re-run this cell and restart
    # the runtime (Runtime -> Restart session), then re-run from the setup cell.
    # Upgrading transformers/peft/trl over Colab's preinstalled versions
    # usually REQUIRES a runtime restart before the new versions are imported.
    # Restart manually (Runtime -> Restart session) OR uncomment the line below
    # to restart automatically, then re-run from the setup cell at the top.
    # import os; os.kill(os.getpid(), 9)
    print("Dependencies installed.")
    print("IMPORTANT: If imports fail or you see a 'RESTART REQUIRED' notice, "
          "restart the runtime (Runtime -> Restart session) and re-run from the setup cell.")
else:
    print("Not on Colab; skipping pip install. Use your local .venv (pip install -r requirements-train.txt).")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 889.0/889.0 kB 49.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 559.1/559.1 kB 47.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 10.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 100.8/100.8 kB 12.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.1/61.1 kB 7.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.1/50.1 MB 52.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 546.2/546.2 kB 48.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 94.3/94.3 kB 11.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 358.3/358.3 kB 37.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 316.1/316.1 kB 34.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 66.4/66.4 kB 7.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.0/84.0 kB 10.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

# Fine-tuned Qwen2.5-Coder-0.5B-Instruct — Step-by-Step Eval

Evaluate the **LoRA fine-tuned** model on AVATAR-TC Java→Python pairs and compare against the pre–fine-tune baseline.

- **Base model:** `Qwen/Qwen2.5-Coder-0.5B-Instruct`
- **Training data:** 1/4 of the AVATAR-TC `train` split (~13,794 pairs), 3 epochs LoRA
- **Eval data:** AVATAR-TC `valid` / `test` (no leakage)

Metrics: **BLEU**, **BERTScore**, **CodeBLEU**, **CodeBERTScore**, and **execution / output-match accuracy** (all via `evaluation/metrics.py` and `evaluation/executor.py`).

After generating Python translations, the notebook runs each prediction and reference in a local subprocess (Colab-safe; no Docker required) and compares stdout.

This reuses the exact same functions as the baseline notebook so the comparison is apples-to-apples.

Set `SHUFFLE=True` and matching `SEED` in both notebooks so eval does not always use the first file-order samples (e.g. `calculateSquareSum`). Use the same `SHUFFLE` / `SEED` / `MAX_SAMPLES` when comparing fine-tuned vs baseline metrics.

## 0. Stage 1 — Java→Python fine-tune (combined corpus)

Builds **`data/processed/java2py_combined`** from:

| Source | Role |
|---|---|
| **AVATAR-TC** (full train) | Contest Java↔Python volume |
| **ClassEval-T** | Class-level OOP |
| **design_patterns_solid** | GoF + SOLID (~1.5k train) |
| **CoDocBench** | Extra Java↔Python if present under `data/raw/codocbench` |

Then LoRA-trains `Qwen/Qwen2.5-Coder-0.5B-Instruct` → `models/java2py_qwen/{adapter,merged}`.

On **Colab**, when Stage 1 finishes the training cell also copies that folder to  
`/content/drive/MyDrive/codegen26_checkpoints/java2py_qwen` so a runtime reset does not lose it.

```bash
python data/scripts/preprocess_java_oop.py --avatar-fraction 1.0 --design-upsample 2
python training/train_java2py.py --config training/configs/java2py_qwen_colab.yaml
```

Flags below default to `False` so "Run all" does not start a long job by accident.


In [ ]:
import sys
from pathlib import Path

try:
    PROJECT_ROOT  # type: ignore[used-before-def]  # noqa: F821
except NameError:
    _here = Path.cwd()
    PROJECT_ROOT = (_here if (_here / "requirements.txt").exists() else _here.parent).resolve()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

# Set True to run (use Colab GPU for full AVATAR).
RUN_PREPROCESS = False
RUN_TRAINING = False

AVATAR_FRACTION = 1.0       # full AVATAR-TC train
DESIGN_UPSAMPLE = 2         # keep patterns visible vs ~55k AVATAR pairs
CONFIG_PATH = PROJECT_ROOT / "training" / "configs" / "java2py_qwen_colab.yaml"
PROCESSED_DIR = PROJECT_ROOT / "data" / "processed" / "java2py_combined"

if RUN_PREPROCESS:
    sys.path.insert(0, str(PROJECT_ROOT / "data" / "scripts"))
    from preprocess_java_oop import build_combined, save_combined  # noqa: E402

    train_pairs, val_pairs = build_combined(
        avatar_fraction=AVATAR_FRACTION,
        include_classeval=True,
        include_design_patterns=True,
        include_codocbench=True,
        design_upsample=DESIGN_UPSAMPLE,
    )
    out = save_combined(train_pairs, val_pairs, output_dir=PROCESSED_DIR)
    print("Stage-1 combined Java2Py dataset saved to", out)
else:
    print("RUN_PREPROCESS=False (skipped). Existing dataset:", PROCESSED_DIR.exists())


In [8]:
!pip uninstall -y torchao
!pip install -U torchao

Found existing installation: torchao 0.10.0
Uninstalling torchao-0.10.0:
  Successfully uninstalled torchao-0.10.0
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.4/3.4 MB 110.2 MB/s eta 0:00:00


In [9]:
# --- Stage 1 training (reuses training/train_base.py) ---
# Writes models/java2py_qwen/{adapter,merged}.
# On Colab, copies the checkpoint to Drive so a runtime reset does not wipe it.
from pathlib import Path
import shutil

STAGE1_DIR = PROJECT_ROOT / "models" / "java2py_qwen"
DRIVE_STAGE1_DIR = Path("/content/drive/MyDrive/codegen26_checkpoints/java2py_qwen")

try:
    IN_COLAB  # noqa: F821
except NameError:
    try:
        import google.colab  # noqa: F401
        IN_COLAB = True
    except ImportError:
        IN_COLAB = False


def save_stage1_to_drive(src: Path = STAGE1_DIR, dst: Path = DRIVE_STAGE1_DIR) -> Path:
    """Persist Stage-1 adapter/merged/checkpoints to Google Drive (Colab only)."""
    if not IN_COLAB:
        print("Not on Colab — skip Drive backup.")
        return src
    if not src.exists():
        raise FileNotFoundError(f"Stage-1 output missing: {src}")
    from google.colab import drive  # type: ignore

    drive.mount("/content/drive", force_remount=False)
    dst.parent.mkdir(parents=True, exist_ok=True)
    shutil.copytree(src, dst, dirs_exist_ok=True)
    print("Stage 1 backed up to Drive →", dst)
    return dst


if RUN_TRAINING:
    sys.path.insert(0, str(PROJECT_ROOT / "training"))
    from train_base import load_config, train  # noqa: E402

    cfg = load_config(str(CONFIG_PATH))
    print("Training with config:", CONFIG_PATH)
    print("dataset_path:", cfg.get("dataset_path"))
    train(cfg)
    print("Stage 1 complete →", STAGE1_DIR / "merged")
    save_stage1_to_drive()
else:
    print("RUN_TRAINING=False (skipped). Set RUN_TRAINING=True above to fine-tune,")
    print("or: python training/train_java2py.py --config training/configs/java2py_qwen_colab.yaml")
    # If /content was wiped but Drive still has today's Stage 1, restore it:
    # save_stage1_to_drive is for backup; restore with the helper below in Stage 2
    # or run: shutil.copytree(DRIVE_STAGE1_DIR, STAGE1_DIR, dirs_exist_ok=True)


Training with config: /content/codegenQwen/codegen26/training/configs/java2py_qwen_colab.yaml
dataset_path: data/processed/java2py_combined
Training on cuda (NVIDIA A100-SXM4-40GB) (dtype=torch.float16)
Precision: fp16=True, bf16=False
Loading tokenizer and model: Qwen/Qwen2.5-Coder-0.5B-Instruct


config.json:   0%|          | 0.00/659 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.30k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

...

Tokenizing train dataset:   0%|          | 0/58263 [00:00<?, ? examples/s]

Building labels for train dataset:   0%|          | 0/58263 [00:00<?, ? examples/s]

Truncating train dataset:   0%|          | 0/58263 [00:00<?, ? examples/s]

Dropping fully masked examples from train dataset:   0%|          | 0/58263 [00:00<?, ? examples/s]

KeyboardInterrupt: 

## Stage 2: Multi-task fine-tune (full mix)

Continues from `models/java2py_qwen/merged`.

If that path is missing after a Colab reconnect, Stage 2 first tries to restore it from  
`/content/drive/MyDrive/codegen26_checkpoints/java2py_qwen`. When Stage 2 finishes it also  
backs up to `/content/drive/MyDrive/codegen26_checkpoints/qwen_multitask`.

**Java2Py:** full AVATAR-TC + ClassEval-T + design_patterns_solid (+ CoDocBench)  
**NL2Py:** MBPP only (Hugging Face; Spider/Bird excluded)  
**Code2Doc:** DocuMint (HF)

Outputs `models/qwen_multitask/{adapter,merged}` then you can push to Hugging Face (next section).

```bash
python data/scripts/preprocess_multitask.py --avatar-fraction 1.0 --design-upsample 2
python training/train_multitask.py --config training/configs/qwen_multitask_colab.yaml
```


In [ ]:
# Stage 2 preprocess
RUN_MULTITASK_PREPROCESS = True
SKIP_HF = False              # MBPP + DocuMint need network
MAX_MBPP = 0                 # 0 = all MBPP
MAX_DOCUMINT = 5000
MAX_COMMENTS = 0             # CodeParrot comments optional
AVATAR_FRACTION = 1.0
DESIGN_UPSAMPLE = 2

MULTITASK_PROCESSED = PROJECT_ROOT / "data" / "processed" / "qwen_multitask"

if RUN_MULTITASK_PREPROCESS:
    sys.path.insert(0, str(PROJECT_ROOT / "data" / "scripts"))
    from preprocess_multitask import collect_all_records, save_multitask_dataset  # noqa: E402

    max_mbpp = None if MAX_MBPP <= 0 else MAX_MBPP
    records = collect_all_records(
        avatar_fraction=AVATAR_FRACTION,
        include_classeval=True,
        include_design_patterns=True,
        include_codocbench=True,
        design_upsample=DESIGN_UPSAMPLE,
        skip_hf=SKIP_HF,
        max_mbpp=max_mbpp,
        max_documint=MAX_DOCUMINT,
        max_comments=MAX_COMMENTS,
    )
    out = save_multitask_dataset(records, output_dir=MULTITASK_PROCESSED)
    print("Multi-task dataset saved to", out)
else:
    print(
        "RUN_MULTITASK_PREPROCESS=False (skipped). Existing dataset:",
        MULTITASK_PROCESSED.exists(),
    )


In [ ]:
# Stage 2 training
RUN_MULTITASK_TRAINING = True

from pathlib import Path
import shutil

STAGE1_DIR = PROJECT_ROOT / "models" / "java2py_qwen"
STAGE1_MERGED = STAGE1_DIR / "merged"
DRIVE_STAGE1_DIR = Path("/content/drive/MyDrive/codegen26_checkpoints/java2py_qwen")
DRIVE_STAGE2_DIR = Path("/content/drive/MyDrive/codegen26_checkpoints/qwen_multitask")
STAGE2_DIR = PROJECT_ROOT / "models" / "qwen_multitask"

try:
    IN_COLAB  # noqa: F821
except NameError:
    try:
        import google.colab  # noqa: F401
        IN_COLAB = True
    except ImportError:
        IN_COLAB = False

MULTITASK_CONFIG = PROJECT_ROOT / "training" / "configs" / (
    "qwen_multitask_colab.yaml" if IN_COLAB else "qwen_multitask_mps.yaml"
)


def restore_stage1_from_drive(
    src: Path = DRIVE_STAGE1_DIR, dst: Path = STAGE1_DIR
) -> bool:
    """If local Stage-1 merged is missing, copy it back from Drive (Colab)."""
    if STAGE1_MERGED.exists():
        return True
    if not IN_COLAB:
        return False
    from google.colab import drive  # type: ignore

    drive.mount("/content/drive", force_remount=False)
    if not (src / "merged").exists():
        return False
    dst.parent.mkdir(parents=True, exist_ok=True)
    shutil.copytree(src, dst, dirs_exist_ok=True)
    print("Restored Stage 1 from Drive →", dst)
    return STAGE1_MERGED.exists()


def save_stage2_to_drive(src: Path = STAGE2_DIR, dst: Path = DRIVE_STAGE2_DIR) -> None:
    if not IN_COLAB or not src.exists():
        return
    from google.colab import drive  # type: ignore

    drive.mount("/content/drive", force_remount=False)
    dst.parent.mkdir(parents=True, exist_ok=True)
    shutil.copytree(src, dst, dirs_exist_ok=True)
    print("Stage 2 backed up to Drive →", dst)


if RUN_MULTITASK_TRAINING:
    if not STAGE1_MERGED.exists() and not restore_stage1_from_drive():
        raise RuntimeError(
            f"Stage-1 merged checkpoint not found at {STAGE1_MERGED} "
            f"(also missing Drive backup {DRIVE_STAGE1_DIR / 'merged'}). "
            "Complete stage 1 (section 0) first, or restore from Drive."
        )
    sys.path.insert(0, str(PROJECT_ROOT / "training"))
    from train_base import load_config, train  # noqa: E402

    cfg = load_config(str(MULTITASK_CONFIG))
    print("Multi-task training with config:", MULTITASK_CONFIG)
    print("dataset_path:", cfg.get("dataset_path"))
    train(cfg)
    print("Stage 2 complete →", STAGE2_DIR / "merged")
    save_stage2_to_drive()
else:
    print("RUN_MULTITASK_TRAINING=False (skipped).")
    print(
        "CLI:",
        f"python training/train_multitask.py --config {MULTITASK_CONFIG.relative_to(PROJECT_ROOT)}",
    )


## Stage 2b — Publish merged model to Hugging Face

After Stage 2 finishes, push `models/qwen_multitask/merged` to the Hub.

1. Create a token with **write** access at https://huggingface.co/settings/tokens  
2. In Colab: Secrets → add `HF_TOKEN`  
3. Set `HF_REPO_ID` below (e.g. `Saikrishna2511/qwen-multitask`) and `RUN_HF_PUSH = True`


In [ ]:
# Push merged multitask checkpoint to Hugging Face Hub
RUN_HF_PUSH = False
HF_REPO_ID = "Saikrishna2511/qwen-multitask"  # change to your namespace/repo
MERGED = PROJECT_ROOT / "models" / "qwen_multitask" / "merged"

if RUN_HF_PUSH:
    if not MERGED.exists():
        raise RuntimeError(f"Merged model not found: {MERGED}. Finish Stage 2 training first.")

    import os
    token = os.environ.get("HF_TOKEN") or os.environ.get("HUGGING_FACE_HUB_TOKEN")
    if not token:
        try:
            from google.colab import userdata  # type: ignore
            token = userdata.get("HF_TOKEN")
        except Exception:
            token = None
    if not token:
        raise RuntimeError(
            "Set HF_TOKEN in the environment or Colab Secrets before pushing."
        )

    from huggingface_hub import HfApi, login
    from transformers import AutoModelForCausalLM, AutoTokenizer

    login(token=token)
    print("Loading merged model from", MERGED)
    tok = AutoTokenizer.from_pretrained(MERGED, trust_remote_code=True)
    model = AutoModelForCausalLM.from_pretrained(MERGED, trust_remote_code=True)
    print("Pushing to", HF_REPO_ID)
    tok.push_to_hub(HF_REPO_ID, token=token)
    model.push_to_hub(HF_REPO_ID, token=token)
    api = HfApi(token=token)
    print("Published:", api.model_info(HF_REPO_ID).id)
else:
    print("RUN_HF_PUSH=False (skipped). After training, set True and provide HF_TOKEN.")


## 1. Imports & path setup

In [ ]:
import sys
from pathlib import Path

try:
    PROJECT_ROOT  # type: ignore[used-before-def]  # noqa: F821
except NameError:
    _here = Path.cwd()
    PROJECT_ROOT = (_here if (_here / "requirements.txt").exists() else _here.parent).resolve()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from inference.generator import CodeGenerator
from evaluation.executor import CodeExecutor
from evaluation.run_baseline_qwen import (
    load_avatar_tc,
    generate_predictions,
    execute_predictions,
    pick_inspect_index,
    compute_oop_breakdown,
)
from evaluation.metrics import (
    compute_bleu,
    compute_bert_score,
    compute_codebleu,
    compute_code_bert_score,
    compute_execution_accuracy,
    compute_output_match_accuracy,
)

print("Project root:", PROJECT_ROOT)

/Users/saikrishnapulupudi/codegenQwen/codegen26/.venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
W0620 11:20:53.764000 18162 .venv/lib/python3.13/site-packages/torch/distributed/elastic/multiprocessing/redirects.py:35] NOTE: Redirects are currently not supported in MacOs.
W0620 11:20:53.800000 18162 .venv/lib/python3.13/site-packages/torch/utils/_pytree.py:630] <enum 'KernelPreference'> is an Enum subclass and is now natively supported by torch.compile as an opaque value type. Calling register_constant() on Enum subclasses is deprecated and will be an error in a future release.
W0620 11:20:53.823000 18162 .venv/lib/python3.13/site-packages/torch/utils/_pytree.py:630] <enum 'ScaleCalculationMode'> is an Enum subclass and is now natively supported by torch.compile as an opaque value type. Calling reg

Project root: /Users/saikrishnapulupudi/codegenQwen/codegen26


## 2. Config

Keep `SPLIT` / `MAX_SAMPLES` / `SHUFFLE` / `SEED` identical to the baseline notebook run so the comparison is fair.

In [ ]:
MODELS_DIR = PROJECT_ROOT / "models" / "java2py_qwen"
MERGED_DIR = MODELS_DIR / "merged"
ADAPTER_DIR = MODELS_DIR / "adapter"
BASE_MODEL = "Qwen/Qwen2.5-Coder-0.5B-Instruct"

SPLIT = "valid"        # "valid" (443) or "test" (1745)
MAX_SAMPLES = 10        # None for full split; match the baseline notebook
BATCH_SIZE = 8
DEVICE = "auto"         # auto -> CUDA on Colab, MPS on Mac, CPU otherwise
SKIP_BERT = False       # True to skip BERTScore / CodeBERTScore
SHUFFLE = True          # randomize which pairs are evaluated when MAX_SAMPLES is set
SEED = 42               # eval subset reproducibility (baseline vs fine-tuned); None = new random batch each run
INSPECT_SEED = None     # preview randomness only; SEED stays fixed for fair eval comparison
INSPECT_INDEX = None    # pin a sample index (overrides INSPECT_SEED), e.g. 3
RUN_EXECUTION = True    # run translated Python after generation
EXEC_TIMEOUT = 10       # seconds per sample
USE_DOCKER = False      # keep False on Colab (no Docker daemon)

# Resolve the fine-tuned model: prefer the merged checkpoint, fall back to the
# raw LoRA adapter. If neither exists, training has not finished yet.
if MERGED_DIR.exists():
    MODEL_PATH, USE_ADAPTER = MERGED_DIR, False
elif ADAPTER_DIR.exists():
    MODEL_PATH, USE_ADAPTER = ADAPTER_DIR, True
else:
    MODEL_PATH, USE_ADAPTER = None, False

if MODEL_PATH is None:
    print(
        "No fine-tuned model found under", MODELS_DIR, "\n\n"
        "Run training first (see section 0), then re-run this cell:\n"
        "  python data/scripts/preprocess_avatar_tc.py\n"
        "  python training/train_java2py.py --config training/configs/java2py_qwen_colab.yaml"
    )
else:
    print(f"Using {'LoRA adapter' if USE_ADAPTER else 'merged'} model:", MODEL_PATH)

Using merged model: /Users/saikrishnapulupudi/codegenQwen/codegen26/models/java2py_qwen/merged


## 3. Load AVATAR-TC data

In [ ]:
data = load_avatar_tc(SPLIT, MAX_SAMPLES, shuffle=SHUFFLE, seed=SEED)
print(f"Loaded {len(data)} samples from AVATAR-TC ({SPLIT})")
print(f"  shuffle={SHUFFLE}, seed={SEED}, unique programs={len({r['java_code'] for r in data})}\n")

PREVIEW_INDEX = (
    INSPECT_INDEX
    if INSPECT_INDEX is not None
    else pick_inspect_index(len(data), seed=INSPECT_SEED)
)

for idx, row in enumerate(data):
    tag = row["java_code"][:70].replace("\n", " ")
    mark = " <-- preview" if idx == PREVIEW_INDEX else ""
    print(f"  [{idx}] {tag}{mark}")

print(f"\n--- Java input (sample {PREVIEW_INDEX}) ---")
print(data[PREVIEW_INDEX]["java_code"][:400])
print(f"\n--- Python reference (sample {PREVIEW_INDEX}) ---")
print(data[PREVIEW_INDEX]["python_code"][:400])

Loaded 10 samples from AVATAR-TC (valid)

--- Java input (sample 1) ---
import java . util . * ; import java . io . * ; import static java . lang . Math . * ; public class Practice { static Scanner scn ; static StringBuilder sb ; public static void main ( String [ ] ScoobyDoobyDo ) { scn = new Scanner ( System . in ) ; sb = new StringBuilder ( ) ; int t = scn . nextInt ( ) ; for ( int tests = 0 ; tests < t ; tests ++ ) solve ( ) ; System . out . println ( sb ) ; } pub

--- Python reference (sample 1) ---
for _ in range ( int ( input ( ) ) ) :
    a , b , c , d = map ( int , input ( ) . split ( ) )
    if ( a == c and b == d ) :
        print ( a , a + 1 )
    elif ( a == c ) :
        print ( a , a + 1 )
    elif ( b == d ) :
        print ( b - 1 , b )
    else :
        print ( a , c )


## 4. Load the fine-tuned model

Loads the merged LoRA checkpoint produced by training.

In [ ]:
if MODEL_PATH is None:
    raise RuntimeError("No fine-tuned model available - run the training step in section 0 first.")

if not USE_ADAPTER:
    # Merged checkpoint: load directly.
    gen = CodeGenerator(model_path=str(MODEL_PATH), device=DEVICE)
else:
    # Adapter-only: load the base model and attach the LoRA adapter.
    from typing import Any, cast

    from transformers import AutoModelForCausalLM, AutoTokenizer
    from peft import PeftModel
    from utils.device import get_best_device, get_inference_dtype

    dev = get_best_device(DEVICE)
    dtype = get_inference_dtype(dev)
    base = AutoModelForCausalLM.from_pretrained(
        BASE_MODEL, torch_dtype=dtype, trust_remote_code=True
    )
    peft_model: Any = cast(Any, PeftModel.from_pretrained(base, str(MODEL_PATH)))
    merged = peft_model.merge_and_unload()

    gen = CodeGenerator.__new__(CodeGenerator)
    gen.model_path = str(MODEL_PATH)
    gen.device = dev
    gen.max_new_tokens = 512
    gen.temperature = 0.2
    gen.top_p = 0.95
    gen.tokenizer = AutoTokenizer.from_pretrained(str(MODEL_PATH), trust_remote_code=True)
    if gen.tokenizer.pad_token is None:
        gen.tokenizer.pad_token = gen.tokenizer.eos_token
    gen.model = merged.to(dev).eval()

print("Loaded:", gen.model_path)

Loading generator on mps (arm) (dtype=torch.bfloat16)


[transformers] `torch_dtype` is deprecated! Use `dtype` instead!
Loading weights: 100%|██████████| 290/290 [00:00<00:00, 53210.91it/s]


Loaded: /Users/saikrishnapulupudi/codegenQwen/codegen26/models/java2py_qwen/merged


## 5. Generate Python translations

In [ ]:
import time

t0 = time.time()
predictions = generate_predictions(data, gen, batch_size=BATCH_SIZE)
gen_time = time.time() - t0

references = [item["python_code"] for item in data]
inputs = [item["java_code"] for item in data]

print(f"\nGeneration: {gen_time:.1f}s ({gen_time / len(data):.2f}s/sample)")

  [8/10] generated
  [10/10] generated

Generation: 38.6s (3.86s/sample)


## 6. Execute generated Python (output-match)

Runs each prediction and its reference in a subprocess, then compares stdout.
Uses local execution (`USE_DOCKER=False`) so this works on Colab without Docker.

In [ ]:
import pandas as pd

if RUN_EXECUTION:
    executor = CodeExecutor(use_docker=USE_DOCKER, timeout=EXEC_TIMEOUT)
    print("Executing predictions...")
    pred_exec = execute_predictions(predictions, executor, batch_size=BATCH_SIZE)
    print("Executing references...")
    ref_exec = execute_predictions(references, executor, batch_size=BATCH_SIZE)
    exec_metrics = {
        **compute_execution_accuracy(pred_exec),
        **compute_output_match_accuracy(pred_exec, ref_exec),
    }
    pd.Series(exec_metrics).round(4)
else:
    pred_exec = ref_exec = None
    print("RUN_EXECUTION=False (skipped)")

Executing predictions...
  [8/10] executed
  [10/10] executed
Executing references...
  [8/10] executed
  [10/10] executed


## 7. BLEU

In [ ]:
bleu = compute_bleu(predictions, references)
bleu

{'bleu': 48.601478514312916,
 'bleu_details': 'BLEU = 48.60 76.3/58.2/47.1/39.8 (BP = 0.905 ratio = 0.909 hyp_len = 1081 ref_len = 1189)'}

## 8. CodeBLEU

In [ ]:
codebleu = compute_codebleu(predictions, references)
codebleu

{'codebleu': 0.5246020195584065,
 'codebleu_ngram': 0.5023728973796341,
 'codebleu_weighted_ngram': 0.5022048466637605,
 'codebleu_syntax': 0.5,
 'codebleu_dataflow': 0.5938303341902313}

## 9. BERTScore (roberta-large)

In [ ]:
if SKIP_BERT:
    bertscore = {"bertscore_f1": None}
    print("Skipped (SKIP_BERT=True)")
else:
    bertscore = compute_bert_score(predictions, references)
bertscore

Loading weights: 100%|██████████| 389/389 [00:00<00:00, 8441.30it/s]
[transformers] RobertaModel LOAD REPORT from: roberta-large
Key                       | Status     | 
--------------------------+------------+-
lm_head.layer_norm.bias   | UNEXPECTED | 
lm_head.dense.weight      | UNEXPECTED | 
lm_head.bias              | UNEXPECTED | 
lm_head.layer_norm.weight | UNEXPECTED | 
lm_head.dense.bias        | UNEXPECTED | 
pooler.dense.weight       | MISSING    | 
pooler.dense.bias         | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


{'bertscore_precision': 0.934411346912384,
 'bertscore_recall': 0.9281848669052124,
 'bertscore_f1': 0.9310976266860962}

## 10. CodeBERTScore (microsoft/codebert-base)

In [ ]:
if SKIP_BERT:
    code_bertscore = {"code_bertscore_f1": None}
    print("Skipped (SKIP_BERT=True)")
else:
    code_bertscore = compute_code_bert_score(predictions, references)
code_bertscore

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 64778.15it/s]


{'code_bertscore_precision': 0.9333752393722534,
 'code_bertscore_recall': 0.9292706251144409,
 'code_bertscore_f1': 0.931179404258728}

## 11. Combined metrics table

In [ ]:
all_metrics = {**bleu, **codebleu, **bertscore, **code_bertscore}
if RUN_EXECUTION and pred_exec is not None:
    all_metrics.update(exec_metrics)

metric_keys = [
    "bleu",
    "bertscore_f1",
    "codebleu",
    "code_bertscore_f1",
    "execution_accuracy",
    "output_match_accuracy",
    "codebleu_ngram",
    "codebleu_weighted_ngram",
    "codebleu_syntax",
    "codebleu_dataflow",
]
summary = pd.Series(
    {k: all_metrics[k] for k in metric_keys if all_metrics.get(k) is not None}
).round(4)
summary

bleu                       48.6015
bertscore_f1                0.9311
codebleu                    0.5246
code_bertscore_f1           0.9312
execution_accuracy          0.4000
output_match_accuracy       0.1000
codebleu_ngram              0.5024
codebleu_weighted_ngram     0.5022
codebleu_syntax             0.5000
codebleu_dataflow           0.5938
dtype: float64

## 12. Baseline vs Fine-tuned comparison

`BASELINE` holds the pre–fine-tune numbers from `baseline_qwen_eval.ipynb`.
Re-run the baseline notebook with the same `SPLIT`, `MAX_SAMPLES`, `SHUFFLE`, and `SEED`, then update these values for a fair comparison.
Execution metrics (`execution_accuracy`, `output_match_accuracy`) are `None` until you run the baseline notebook with `RUN_EXECUTION=True`.

In [ ]:
BASELINE = {
    "bleu": 33.2392,
    "codebleu": 0.3883,
    "bertscore_f1": 0.8821,
    "code_bertscore_f1": 0.8861,
    "execution_accuracy": None,
    "output_match_accuracy": None,
}

compare_keys = [
    "bleu",
    "codebleu",
    "bertscore_f1",
    "code_bertscore_f1",
    "execution_accuracy",
    "output_match_accuracy",
]
rows = []
for k in compare_keys:
    base = BASELINE.get(k)
    ft = all_metrics.get(k)
    delta = (ft - base) if (base is not None and ft is not None) else None
    rows.append({
        "metric": k,
        "baseline": round(base, 4) if base is not None else None,
        "fine_tuned": round(ft, 4) if ft is not None else None,
        "delta": round(delta, 4) if delta is not None else None,
    })

comparison = pd.DataFrame(rows).set_index("metric")
comparison

,baseline,fine_tuned,delta
metric,,,
bleu,33.2392,48.6015,15.3623
codebleu,0.3883,0.5246,0.1363
bertscore_f1,0.8821,0.9311,0.0490
code_bertscore_f1,0.8861,0.9312,0.0451
execution_accuracy,NaN,0.4000,NaN
output_match_accuracy,NaN,0.1000,NaN


## 13. OOP category breakdown (optional)

In [ ]:
oop_breakdown = compute_oop_breakdown(predictions, references, inputs)
pd.DataFrame(oop_breakdown).T

,count,codebleu,syntax,dataflow
general,9.0,0.5343,0.498,0.6162
design_patterns,1.0,0.4112,0.520,0.3438


## 14. Inspect a prediction vs reference

In [ ]:
i = PREVIEW_INDEX
print(f"=== Sample {i} ===")
print("=== Java input ===")
print(inputs[i][:500])
print("\n=== Prediction (fine-tuned) ===")
print(predictions[i][:500])
print("\n=== Reference ===")
print(references[i][:500])

if RUN_EXECUTION and pred_exec is not None and ref_exec is not None:
    pred_result = pred_exec[i]
    ref_result = ref_exec[i]
    stdout_match = (
        pred_result.get("passed")
        and ref_result.get("passed")
        and pred_result.get("stdout", "").strip() == ref_result.get("stdout", "").strip()
    )
    print("\n=== Prediction stdout ===")
    print(pred_result.get("stdout", "") or "(empty)")
    if pred_result.get("stderr"):
        print("\n=== Prediction stderr ===")
        print(pred_result["stderr"])
    if pred_result.get("error"):
        print("\n=== Prediction error ===")
        print(pred_result["error"])
    print("\n=== Reference stdout ===")
    print(ref_result.get("stdout", "") or "(empty)")
    if ref_result.get("stderr"):
        print("\n=== Reference stderr ===")
        print(ref_result["stderr"])
    if ref_result.get("error"):
        print("\n=== Reference error ===")
        print(ref_result["error"])
    print(
        f"\n=== Match: {stdout_match} | "
        f"Passed: pred={pred_result.get('passed')} ref={ref_result.get('passed')} ==="
    )
else:
    print("\n(Execution skipped — set RUN_EXECUTION=True and re-run section 6)")

=== Sample 1 ===
=== Java input ===
import java . util . * ; import java . io . * ; import static java . lang . Math . * ; public class Practice { static Scanner scn ; static StringBuilder sb ; public static void main ( String [ ] ScoobyDoobyDo ) { scn = new Scanner ( System . in ) ; sb = new StringBuilder ( ) ; int t = scn . nextInt ( ) ; for ( int tests = 0 ; tests < t ; tests ++ ) solve ( ) ; System . out . println ( sb ) ; } public static void solve ( ) { int l1 = scn . nextInt ( ) , r1 = scn . nextInt ( ) ; int l2 = scn . nex

=== Prediction (fine-tuned) ===
t = int ( input ( ) )
for _ in range ( t ) :
    a , b = map ( int , input ( ) . split ( ) )
    c , d = map ( int , input ( ) . split ( ) )
    for i in range ( a , b + 1 ) :
        for j in range ( c , d + 1 ) :
            if i == j : print ( i , j )
            else : pass

=== Reference ===
for _ in range ( int ( input ( ) ) ) :
    a , b , c , d = map ( int , input ( ) . split ( ) )
    if ( a == c and b == d ) :
      